# Week 19, Day 3: Monte Carlo and the Metropolis algorithm

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w019/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Write a Metropolis sampler

Write `metropolis(logp, x0, steps, width, rng)` returning an array of the chain's states (one row per step) and the acceptance rate. Sample 20,000 steps from a two-dimensional Gaussian with correlation 0.9, starting at (−3, 3) with width 0.5, and check the moments after discarding 1,000 burn-in steps.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1953)

def metropolis(logp, x0, steps, width, rng):
    x = np.array(x0, dtype=float); lp = logp(x)
    out = np.empty((steps, len(x))); accepted = 0
    for t in range(steps):
        prop = ...
        lp_prop = logp(prop)
        if ...:
            x, lp = prop, lp_prop
            accepted += 1
        out[t] = x
    return out, accepted / steps

rho = 0.9
Sinv = np.linalg.inv(np.array([[1, rho], [rho, 1]]))
def log_target(x): return -0.5 * x @ Sinv @ x
samples, acc = metropolis(log_target, [-3.0, 3.0], 20000, 0.5, rng)
kept = samples[1000:]
print(f"acceptance {acc:.2f};  mean {kept.mean(axis=0).round(3)};  correlation {np.corrcoef(kept.T)[0, 1]:.3f}")
plt.plot(samples[:500, 0], samples[:500, 1], ".-", ms=2, lw=0.5); plt.axis("equal"); plt.show()

In [ ]:
import numpy as np
def test_shape():
    "metropolis returns (samples, acceptance rate)"
    s, acc = metropolis(lambda x: -0.5 * np.sum(x ** 2), np.zeros(2), 100, 1.0, np.random.default_rng(0))
    assert s.shape == (100, 2) and 0 <= acc <= 1

def test_moments():
    "after burn-in, the samples of the correlated Gaussian have means near 0 and correlation near 0.9"
    kept = samples[1000:]
    assert np.all(np.abs(kept.mean(axis=0)) < 0.15) and abs(np.corrcoef(kept.T)[0, 1] - 0.9) < 0.05

test_shape()
test_moments()
print("All checks passed")

### Exercise: Choose the proposal width

Write `ess(x)` for a one-dimensional chain: compute autocorrelations up to lag 200 and sum them until the first one below 0.05. For widths 0.05, 1 and 8, run 5,000 steps on the correlated Gaussian, discard 500, and store the acceptance rates in `acc_by_width` and the ESS of the first coordinate in `ess_by_width`.

In [ ]:
def ess(x, max_lag=200):
    x = x - x.mean()
    var = np.mean(x ** 2)
    total = 0.0
    for k in range(1, max_lag):
        rho_k = ...
        if rho_k < 0.05:
            break
        total += rho_k
    return len(x) / (1 + 2 * total)

acc_by_width, ess_by_width = {}, {}
for w in [0.05, 1.0, 8.0]:
    s, a = metropolis(log_target, [0.0, 0.0], 5000, w, rng)
    acc_by_width[w] = a
    ess_by_width[w] = ...
print("acceptance:", {w: round(a, 2) for w, a in acc_by_width.items()}); print("effective sample size of 4,500 kept steps:", {w: round(e) for w, e in ess_by_width.items()})

In [ ]:
import numpy as np
def test_ess_fn():
    "ess of independent samples is close to their number"
    x = np.random.default_rng(1).normal(size=4000)
    assert 3000 < ess(x) < 5000

def test_best_middle():
    "the intermediate width has the largest effective sample size of the three"
    assert ess_by_width[1.0] > ess_by_width[0.05] and ess_by_width[1.0] > ess_by_width[8.0]

test_ess_fn()
test_best_middle()
print("All checks passed")

### Exercise: Credible intervals for a preferred orientation

With 60 trials, write `log_post(w)` (Poisson log-likelihood plus a N(0, 3²) log-prior on each weight), run Metropolis for 20,000 steps (width 0.15) from zero, discard 2,000, convert each sample to a preferred orientation in degrees, and store its 95% credible interval in `ci_pref`.

In [ ]:
r_t = np.random.default_rng(60)
theta = r_t.uniform(0, np.pi, 60)
X = np.column_stack([np.ones(60), np.cos(2 * theta), np.sin(2 * theta)])
k = r_t.poisson(np.exp(1 + 0.8 * np.cos(2 * (theta - np.deg2rad(60)))))

def log_post(w):
    eta = X @ w
    return ...

chain, acc = metropolis(log_post, np.zeros(3), 20000, 0.15, r_t)
W_s = chain[2000:]
pref = ...                                         # degrees between 0 and 180, one per sample
ci_pref = np.percentile(pref, [2.5, 97.5])
print(f"acceptance {acc:.2f};  posterior mean weights {W_s.mean(axis=0).round(2)};  preferred orientation 95% interval {ci_pref.round(1)}")
plt.hist(pref, 60); plt.axvline(60, color="k"); plt.xlabel("preferred orientation (deg)"); plt.show()

In [ ]:
import numpy as np
def test_logpost():
    "log_post is the Poisson log-likelihood plus the Gaussian log-prior (up to a constant)"
    w = np.array([0.5, 0.2, -0.1]); eta = X @ w
    assert abs(log_post(w) - (np.sum(k * eta - np.exp(eta)) - np.sum(w ** 2) / (2 * 3 ** 2))) < 1e-9

def test_interval():
    "the 95% credible interval for the preferred orientation contains the true 60 degrees and is narrower than 30 degrees"
    assert ci_pref[0] < 60 < ci_pref[1] and ci_pref[1] - ci_pref[0] < 30

test_logpost()
test_interval()
print("All checks passed")